<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/HW/OwenAbbata/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
def solve_bridge_circuit(V0, R1, R2, R3, R4, R5):
  """Solve a bridge circuit with five resistors and one voltage source.

    The circuit has the topology:
        B ---- R4 ---- C--(-)
        | L            |
        |  L           |
        R3  L__R5 _    R2
        |           L  |
        |            L |
    (+)-A ---- R1 ---- D


    Uses Kirchhoff's current and voltage laws to set up
    a system of linear equations A @ I = b.

    Parameters
    ----------
    V0 : float
        Voltage of the source (V).
    R1, R2, R3, R4, R5 : float
        Resistances (Ohm).

    Returns
    -------
    currents : numpy.ndarray
        Array [I1, I2, I3, I4, I5] of branch currents (A).
    """
   #Assume current directions:
   #I1: V0 -> R1 -> Node B
   #I2: Node B -> R2 -> Node C (Ground)
   #I3: V0 -> R3 -> Node D
   #I4: Node D -> R4 -> Node C (Ground)
   #I5: Node B -> R5 -> Node D

   #Formulate equations based on KCL at nodes B and D, and KVL for three independent loops.
   #Equations:
   #1. KCL at Node B: I1 - I2 - I5 = 0
   #2. KCL at Node D: I3 + I5 - I4 = 0
   #3. KVL loop V0-R1-R2-C-V0: V0 - I1*R1 - I2*R2 = 0  => R1*I1 + R2*I2 = V0
   #4. KVL loop V0-R3-R4-C-V0: V0 - I3*R3 - I4*R4 = 0  => R3*I3 + R4*I4 = V0
   #5. KVL loop B-R5-D-R4-R2-B: I5*R5 + I4*R4 - I2*R2 = 0 => -R2*I2 + R4*I4 + R5*I5 = 0


  A = np.array([
      [1, -1, 0, 0, -1],  # KCL at Node B (I1 - I2 - I5 = 0)
      [0, 0, 1, -1, 1],   # KCL at Node D (I3 - I4 + I5 = 0)
      [R1, R2, 0, 0, 0],  # KVL Loop A-B-C-A (R1*I1 + R2*I2 = V0)
      [0, 0, R3, R4, 0],  # KVL Loop A-D-C-A (R3*I3 + R4*I4 = V0)
      [0, -R2, 0, R4, R5] # KVL Loop B-C-D-B (-R2*I2 + R4*I4 + R5*I5 = 0)
  ], dtype=float)

  b = np.array([0, 0, V0, V0, 0], dtype=float)

  print("Matrix A:")
  print(A)
  print("Matrix b:")
  print(b)

  currents = np.linalg.solve(A, b)
  return currents


# Given Values
V0 = 10  # Volts
R1 = 100  # Ohm
R2 = 200  # Ohm
R3 = 300  # Ohm
R4 = 600  # Ohm
R5 = 50   # Ohm

currents = solve_bridge_circuit (V0, R1, R2, R3, R4, R5)

print("Branch currents:")
print(f"I1 = {currents[0]:.4f} A  (through R1)")
print(f"I2 = {currents[1]:.4f} A  (through R2)")
print(f"I3 = {currents[2]:.4f} A  (through R3)")
print(f"I4 = {currents[3]:.4f} A  (through R4)")
print(f"I5 = {currents[4]:.4f} A  (through R5)")

Matrix A:
[[   1.   -1.    0.    0.   -1.]
 [   0.    0.    1.   -1.    1.]
 [ 100.  200.    0.    0.    0.]
 [   0.    0.  300.  600.    0.]
 [   0. -200.    0.  600.   50.]]
Matrix b:
[ 0.  0. 10. 10.  0.]
Branch currents:
I1 = 0.0333 A  (through R1)
I2 = 0.0333 A  (through R2)
I3 = 0.0111 A  (through R3)
I4 = 0.0111 A  (through R4)
I5 = 0.0000 A  (through R5)


In [ ]:
def verify_power_balance(V0, R1, R2, R3, R4, R5, currents):
  """Verify conservation of energy in the circuit.

    Checks that the power delivered by the voltage source equals the total power dissipated in all resistors.

    Parameters
    ----------
    V0 : float
        Voltage of the source (V).
    R1, R2, R3, R4, R5 : float
        Resistances (Ohm).
    currents : numpy.ndarray
        Array [I1, I2, I3, I4, I5] of branch currents (A).

    Returns
    -------
    bool
        True if power balance holds (within numerical tolerance).
    """
  I1, I2, I3, I4, I5 = currents
  P_source = V0 * (I1 + I3)
  P_dissipated = I1**2 * R1 + I2**2 * R2 + I3**2 * R3 + I4**2 * R4 + I5**2 * R5

  print(f"Power delivered by source:     {P_source:.4f} W")
  print(f"Power dissipated in resistors: {P_dissipated:.4f} W")

  return np.isclose(P_source, P_dissipated)


balanced = verify_power_balance(V0, R1, R2, R3, R4, R5, currents)
print(f"Energy conserved: {balanced}")

Power delivered by source:     0.4444 W
Power dissipated in resistors: 0.4444 W
Energy conserved: True


In [ ]:
def verify_bridge_balance( R1, R2, R3, R4 ):
  """Verify circuit bridge is balanced.

    Checks that no current is flowing in R5 by seeing is R1/R3 = R2/R4 if R3 = 0 or R4 = 0 it will be set to false

    Parameters
    ----------
    R1, R2, R3, R4: float
        Resistances (Ohm).


    Returns
    -------
    bool
        True if bridge balance holds (within numerical tolerance).
    """
  if R3 == 0 or R4 == 0:
    print("Cannot determine balance if R3 or R4 is zero.")
    return False

  is_balanced = np.isclose(R1 / R3, R2 / R4)
  print(f"Bridge is balanced: {is_balanced}")
  return is_balanced

balanced_check = verify_bridge_balance(R1, R2, R3, R4)


Bridge is balanced: True


Part 2

I am using ChatGPT as my AI tool with its code writen below:

1.How your AI model handled the physics (did it get the equations right on the first try?)
 - It didnt handle it well on the first try and took many cleficications for the AI to give me any useful code


2.How many iteration steps it needed to arrive at a physically realistic circuit
- To get to the code below it took 4 prompts to help make a realisitic circuit

3.Where in nature or engineering you would find the final circuit
- Inductor circuits are found in motors

In [1]:
import numpy as np

def solve_bridge(f, R1, R2, R3, R4, R5, L, Vin):
    omega = 2*np.pi*f
    j = 1j

    # Impedances
    Z1 = R1
    Z2 = R2
    Z3 = R3
    Z4 = R4
    Z5 = R5 + j*omega*L   # <-- inductor added here

    # Admittances
    Y1 = 1/Z1
    Y2 = 1/Z2
    Y3 = 1/Z3
    Y4 = 1/Z4
    Y5 = 1/Z5

    # Unknown nodes: VB and VD
    A = np.array([
        [Y1 + Y2 + Y5, -Y5],
        [-Y5, Y3 + Y4 + Y5]
    ], dtype=complex)

    b = np.array([
        Y1*Vin,
        Y4*Vin
    ], dtype=complex)

    VB, VD = np.linalg.solve(A, b)

    return VB - VD

In [2]:
frequencies = np.logspace(1, 5, 500)
Vout = np.array([solve_bridge(f, 1000,1000,1000,1000,100,0.01,1)
                 for f in frequencies])

In [3]:
frequencies = np.logspace(1, 5, 20)  # fewer points so printing is readable

for f in frequencies:
    Vout = solve_bridge(f, 1000,1000,1000,1000,100,0.01,1)
    print(f"f = {f:8.1f} Hz | Vout = {Vout}")

f =     10.0 Hz | Vout = 5.421010862427522e-20j
f =     16.2 Hz | Vout = 5.421010862427522e-20j
f =     26.4 Hz | Vout = -2.168404344971009e-19j
f =     42.8 Hz | Vout = 0j
f =     69.5 Hz | Vout = 1.3010426069826053e-18j
f =    112.9 Hz | Vout = 1.734723475976807e-18j
f =    183.3 Hz | Vout = 1.734723475976807e-18j
f =    297.6 Hz | Vout = 1.1275702593849246e-17j
f =    483.3 Hz | Vout = (-1.1102230246251565e-16-1.5612511283791264e-17j)
f =    784.8 Hz | Vout = -2.0816681711721685e-17j
f =   1274.3 Hz | Vout = -2.42861286636753e-17j
f =   2069.1 Hz | Vout = 2.0816681711721685e-17j
f =   3359.8 Hz | Vout = -2.7755575615628914e-17j
f =   5455.6 Hz | Vout = 2.7755575615628914e-17j
f =   8858.7 Hz | Vout = 2.7755575615628914e-17j
f =  14384.5 Hz | Vout = 2.7755575615628914e-17j
f =  23357.2 Hz | Vout = (5.551115123125783e-17+2.7755575615628914e-17j)
f =  37926.9 Hz | Vout = (1.6653345369377348e-16+1.3877787807814457e-17j)
f =  61584.8 Hz | Vout = (5.551115123125783e-17+4.163336342344337e-